In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import pickle
import seaborn as sns
from scipy.stats import ttest_ind, ttest_ind_from_stats
pd.set_option('display.max_columns', None)
import scipy.stats as stats

In [2]:
from numpy.polynomial.polynomial import polyfit

In [3]:
import pkg_resources
pkg_resources.require("scikit-learn==0.20.2")
import sklearn
print(sklearn.__file__)
print(sklearn.__version__)

ModuleNotFoundError: No module named 'pkg_resources'

In [ ]:
# replace this with simulated data
filepath = "./model_input_sample_small_train.txt"

In [ ]:
df = pd.read_csv(filepath, delimiter= '\s+')

In [ ]:
actions=df.iloc[:,(2+375+1):(2+375+1+25)]

In [ ]:
action_recategorize = pd.read_csv('./Action_Recategorize_simulate.csv')
action_text=actions[actions==1].stack().reset_index().drop(0,1)
# need to change variable names with simulated data
temp =pd.concat([df[["buyer_id","receive_time_id","host_liv_stat_rate_sum","buyer_stat_dim_tb_mbr_tq_score","div_pay_amt_fillna"]],action_text],axis=1)
temp1=temp.merge(action_recategorize,left_on='level_1',right_on='text')
temp2=temp1.sort_values(['buyer_id','receive_time_id'])

In [ ]:
temp2['jian_lag'] = temp2.groupby('buyer_id')['jian_cat'].shift(1)
temp2['man_lag'] = temp2.groupby('buyer_id')['man_cat'].shift(1)
temp2['jian_lead'] = temp2.groupby('buyer_id')['jian_cat'].shift(-1)
temp2['man_lead'] = temp2.groupby('buyer_id')['man_cat'].shift(-1)
temp2['rating_lag'] = temp2.groupby('buyer_id')['host_liv_stat_rate_sum'].shift(1)
temp2['revisit'] = np.where(temp2['host_liv_stat_rate_sum']== temp2['rating_lag'], 1, 0)
temp2['buy']=np.sign(temp2['div_pay_amt_fillna'])
temp2['buy_lag'] = temp2.groupby('buyer_id')['buy'].shift(1)

# 1. Figure 2 Redemption Rate

In [ ]:
df_jian = temp2.groupby('jian_cat').agg({'buy': ['mean', 'sem']}).xs('buy', axis=1, drop_level=True).reset_index('jian_cat')
df_man = temp2.groupby('man_cat').agg({'buy': ['mean', 'sem']}).xs('buy', axis=1, drop_level=True).reset_index('man_cat')

## Fig 2(a)

In [ ]:
x_ticklabels = ['LL', 'L', 'M', 'H', 'HH']
palette = sns.color_palette(['#7428a6'])
fontsize=12
sns.set(font='Times New Roman', style="white")
fontsize = 18
fig, ax = plt.subplots(figsize=(6, 6))
sns.barplot(x=df_jian["jian_cat"],y=df_jian["mean"], ax=ax,palette=palette)
#plt.errorbar(x=df_jian["jian_cat"]-1,y=df_jian["mean"],yerr=df_jian["std"], fmt='none',c ="black",elinewidth=3, capsize=10, capwidth=30)
plt.errorbar(x=df_jian["jian_cat"]-1,y=df_jian["mean"],yerr=df_jian["sem"], fmt='none',c ="black",elinewidth=3, capsize=10)
# ax.set(ylim=(0.09, 0.11))
ax.tick_params(labelsize=fontsize)
ax.set_xticklabels(x_ticklabels)
# ax.set_title('Coupon XXX', fontsize=fontsize)
ax.set_xlabel('Discount Level', fontsize=fontsize, fontweight="bold",fontname ='Times New Roman')
ax.set_ylabel('Average Redemption Rate', fontsize=fontsize, fontweight="bold",fontname ='Times New Roman')
plt.tight_layout()
#plt.show()
plt.savefig('./By-Discount-Level.pdf')

## Figure 2(b)

In [ ]:
x_ticklabels = ['LL', 'L', 'M', 'H', 'HH']
palette = sns.color_palette(['#7428a6'])
sns.set(font="Times New Roman", style="white")
fontsize = 18
fig, ax = plt.subplots(figsize=(6, 6))
sns.barplot(x=df_man["man_cat"],y=df_man["mean"], ax=ax,palette=palette)
#plt.errorbar(x=df_jian["jian_cat"]-1,y=df_jian["mean"],yerr=df_jian["std"], fmt='none',c ="black",elinewidth=3, capsize=10, capwidth=30)
plt.errorbar(x=df_man["man_cat"]-1,y=df_man["mean"],yerr=df_man["sem"], fmt='none',c ="black",elinewidth=3, capsize=10)
# ax.set(ylim=(0.09, 0.11))
ax.tick_params(labelsize=fontsize)
ax.set_xticklabels(x_ticklabels)
# ax.set_title('Coupon XXX', fontsize=fontsize)
ax.set_xlabel('Threshold Level', fontsize=fontsize, fontweight="bold",fontname ='Times New Roman')
ax.set_ylabel('Average Redemption Rate', fontsize=fontsize, fontweight="bold",fontname ='Times New Roman')
plt.tight_layout()
#plt.show()
plt.savefig('./By-Threshold-Level.pdf')

# 2. Figure 3 Heterogeneity

In [ ]:
temp2.buyer_stat_lbs_consume_level.quantile([.1, .5, .9])

In [ ]:
temp2.loc[temp2['buyer_stat_lbs_consume_level'] <= -0.891503, 'Spend level'] = 'Low Spenders' 
temp2.loc[temp2['buyer_stat_lbs_consume_level'] > 1.121342, 'Spend level'] = 'High Spenders'
temp4 = temp2.groupby(['Spend level','jian_cat']).agg({'div_pay_amt_fillna': ['mean', 'sem']}).xs('div_pay_amt_fillna', axis=1, drop_level=True).reset_index(['Spend level','jian_cat'])

In [ ]:
d = {1:"LL",2:"L",3:"M",4:"H",5:"HH"}
temp4["Discount level"] = temp4["jian_cat"].apply(lambda x:d[x])

In [ ]:
#Only two bars
temp_d2 = temp4[temp4["Discount level"].isin(["L","H"])]

In [ ]:
sns.set(font="Times New Roman", style="white")
palette = sns.color_palette(['#7428a6', '#fb98dd'])
fig, ax = plt.subplots(figsize=(5, 5),dpi=100)
fontsize = 18
g = sns.barplot(x='Spend level', y='mean', 
                hue="Discount level", data=temp_d2, ax=ax, palette=palette)

ax.tick_params(labelsize=fontsize)
ax.set_xlabel('Buyer Consumption Level', fontsize=fontsize,fontweight="bold",fontname ='Times New Roman')
ax.set_ylabel('Average Revenue', fontsize=fontsize,fontweight="bold",fontname ='Times New Roman')
# ax.set_xticklabels(x_ticklabels)

leg = ax.legend(fontsize=fontsize)
new_title = "Discount Level"
leg.set_title(new_title,prop={'size':fontsize})
plt.tight_layout()
# plt.show()
plt.savefig('./GMV-and-Coupon_LH_paper.pdf')

# Figure 4. Reference Price

## t= M

In [ ]:
temp_1 = temp2[temp2['jian_cat']==3]
temp_1['period'] = "RedemptionRate_M_t+1"
temp_2 = temp2[temp2['jian_lead']==3]
temp_2['period'] = "RedemptionRate_t"
frames = [temp_1, temp_2]
temp_combine = pd.concat(frames)
temp_combine = temp_combine.sort_values(by=['buyer_id', 'receive_time_id'])

In [ ]:
temp_combine['coupon_t-1'] = np.where(temp_combine['period']== 'RedemptionRate_t', temp_combine['jian_cat'], temp_combine['jian_lag'])
tempplot = temp_combine.groupby(['coupon_t-1','period']).agg({'buy': ['mean', 'sem']}).xs('buy', axis=1, drop_level=True).reset_index(['coupon_t-1','period']).sort_values(['coupon_t-1', 'period'], ascending=[True, False])

In [ ]:
#Only two bars
temp_d2 = tempplot[tempplot["coupon_t-1"].isin([2,4])]

In [ ]:
x_ticklabels = [ 'L', 'H']
sns.set(font="Times New Roman", style="white")
palette = sns.color_palette(['#fb98dd','#7428a6'])
fig, ax = plt.subplots(figsize=(5, 5),dpi=100)
sns.barplot(x='coupon_t-1', y='mean', 
                hue="period", data=temp_d2, ax=ax, palette=palette)

ax.tick_params(labelsize=fontsize)
ax.set_xlabel('Coupon Discount Level t', fontsize=fontsize,fontweight="bold",fontname ='Times New Roman')
ax.set_ylabel('Redemption Rate t+1', fontsize=fontsize,fontweight="bold",fontname ='Times New Roman')
ax.set_xticklabels(x_ticklabels)

leg = ax.legend(fontsize=fontsize)
new_title = ""
leg.set_title(new_title,prop={'size':fontsize})
#plt.show()
plt.savefig('./ref_price_LH_paper.pdf')

# Figure 5. State Dependence

In [ ]:
# Select IDs two consecutive H coupons
temp_1 = temp2[(temp2['jian_cat']==4) & (temp2['jian_lag']==4)]
temp_1['period'] = "t"
temp_2 = temp2[(temp2['jian_lead']==4) & (temp2['jian_cat']==4)]
temp_2['period'] = "t-1"
frames = [temp_1, temp_2]
temp_combine = pd.concat(frames)
temp_combine = temp_combine.sort_values(by=['buyer_id', 'receive_time_id'])

In [ ]:
lst = [['No_Purchase_t-1', temp_combine.buy[temp_combine["buy_lag"] == 0].mean(),temp_combine.buy[temp_combine["buy_lag"] == 0].sem()], 
       ['Purchase_t-1'   , temp_combine.buy[temp_combine["buy_lag"] == 1].mean(),temp_combine.buy[temp_combine["buy_lag"] == 1].sem()]      ]   
vs = pd.DataFrame(lst, columns =['label', 'mean','sem'])

In [ ]:
palette = sns.color_palette(['#7428a6'])
sns.set(font="Times New Roman", style="white")
fontsize = 18
fig, ax = plt.subplots(figsize=(5, 5))
ax.set(ylim=(0, 0.12))
sns.barplot(x=vs["label"],y=vs["mean"], ax=ax,palette=palette)
plt.errorbar(x=vs["label"],y=vs["mean"],yerr=vs["sem"], fmt='none',c ="black",elinewidth=3, capsize=10)
ax.tick_params(labelsize=fontsize)
ax.set_xlabel('', fontsize=fontsize, fontweight="bold",fontname ='Times New Roman')
ax.set_ylabel('Pruchase_Likelihood_t', fontsize=fontsize, fontweight="bold",fontname ='Times New Roman')
plt.tight_layout()
#plt.show()
plt.savefig('./variety_seeking_HH_paper.pdf')

# Figure 6: TQZ vs GMV

In [ ]:
# std and mean of two features
std_dict= {'prod_stat_avg_reserve_price':158611.8316,'buyer_stat_dim_tb_mbr_tq_score':447.1282046}
mean_dict= {'prod_stat_avg_reserve_price':5118.297942,'buyer_stat_dim_tb_mbr_tq_score':946.6162365}

In [ ]:
modelpathBENCHMARK = "./Benchmark_saved_model/"
#modelnameLR = "LinearRegression1125.sav"
modelnameGBDT = "gbdt_best_estimator_regression.sav"
#modelnameORF = "orf_saved.pkl"
#modelnameNN = "nn_model.pt"

#if model_name =='GBDT':
pretrained_model_path= os.path.join(modelpathBENCHMARK,modelnameGBDT)
#pretrained_model_path='./gbdt_best_estimator_regression.sav'
print(pretrained_model_path)

model_dict = pickle.load(open(pretrained_model_path, 'rb'))
model = model_dict['model']
num_state_var = model_dict['num_state_var']
num_action = model_dict['num_action']

In [ ]:
data_modified = df.copy()
states = data_modified.iloc[:, 2:num_state_var+2]
states = states.values
action = data_modified.iloc[:, num_state_var+2:num_state_var+2+num_action]
action =action.values
X = np.concatenate([states, action], axis=1)
y_pred = model.predict(X)

In [ ]:
data_predict = data_modified[['buyer_stat_dim_tb_mbr_tq_score','div_pay_amt_fillna']]
# transform back to real value before standardize
data_predict['prod_stat_avg_reserve_price'] = data_predict['buyer_stat_dim_tb_mbr_tq_score']*std_dict['buyer_stat_dim_tb_mbr_tq_score']+mean_dict['buyer_stat_dim_tb_mbr_tq_score']
# rename columns
data_predict['GMV predicted'] = y_pred
data_predict['GMV observed'] = data_predict['div_pay_amt_fillna']

In [ ]:
font = {'fontname':'Times New Roman'}
fig,ax=plt.subplots(figsize=(7,4))
fontsize=12
df_plot = data_predict.copy()
df_plot['tqz_binned'] = pd.qcut(df_plot['buyer_stat_dim_tb_mbr_tq_score'], 10, precision=1)
#sns.barplot(x="GMV predicted", y="tqz_binned", data=df_plot)
sns.barplot(x="tqz_binned", y="GMV predicted", data=df_plot, ci=99, errwidth=1,capsize=0.1,palette="Purples")   
ax.set_xlabel('TQZ Decile',fontsize=fontsize, **font)
ax.set_ylabel('GMV Predicted',fontsize=fontsize,**font)
ax.tick_params(labelsize=12)
for tick in ax.get_xticklabels():
    tick.set_fontname('Times New Roman')
for tick in ax.get_yticklabels():
    tick.set_fontname('Times New Roman')
plt.xticks(rotation=45)
ax.set_ylim([0,8])
plt.tight_layout()
plt.savefig('./model_bias_TQZ_predict.pdf')
plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
#sns.barplot(x="GMV observed", y="tqz_binned", data=df_plot)
sns.barplot(x="tqz_binned", y="GMV observed", data=df_plot, capsize=0.1, errwidth=1, ci=95,palette="Purples")
ax.set_xlabel('TQZ Decile',fontsize=fontsize,**font)
ax.set_ylabel('GMV Observed',fontsize=fontsize,**font)
ax.tick_params(labelsize=12)
plt.xticks(rotation=45)
for tick in ax.get_xticklabels():
    tick.set_fontname("Times New Roman")
for tick in ax.get_yticklabels():
    tick.set_fontname("Times New Roman")
ax.set_ylim([0,8])
plt.tight_layout()
plt.savefig('model_bias_TQZ_real.pdf')
plt.show()

# Figure 9: History of CLV Gain: see R

# Figure 10: Actual vs Predicted CLV Gain: see R

# Figure 11: Targeting Rule Under BDRL: Host Attractiveness

In [ ]:
df_bcq = pd.read_csv("./BCQ_distribution_2_timeid.csv",index_col=0)

In [ ]:
df_real = pd.read_csv("./fans_value_jian_buyer_download.csv",header=None)

In [ ]:
df_real.columns = ["buyer_id","sequence_number", "attractive_level"]

In [ ]:
df_ref = pd.read_csv("./action_type_reference_withPos_3.csv",index_col=0)

In [ ]:
df_reference = df_ref[["action_pos_type","jian_level"]]

In [ ]:
df_bcq_jian = df_bcq.merge(df_reference, left_on="BCQ_Action", right_on="action_pos_type")

In [ ]:
df_bcq_jian = df_bcq_jian[["mdp_id","sequence_number","jian_level"]].rename(columns={"mdp_id":"buyer_id","jian_level":"BCQ_jian_level"})

In [ ]:
df_bcq_jian = df_bcq_jian.merge(df_real, on=["buyer_id","sequence_number"])

In [ ]:
# find median
import statistics
statistics.median(df_bcq_jian.attractive_level)

In [ ]:
df_bcq_jian.loc[df_bcq_jian['attractive_level'] <= 740, 'Attractive level'] = 'Less Attractive'
df_bcq_jian.loc[df_bcq_jian['attractive_level']  > 740, 'Attractive level'] = 'More Attractive'

In [ ]:
df_bcq_jian=df_bcq_jian[["Attractive level","BCQ_jian_level","buyer_id"]]

In [ ]:
bcq_dist = df_bcq_jian.groupby(["Attractive level","BCQ_jian_level"]).count().reset_index().rename(columns={"buyer_id":"cnt"})

In [ ]:
palette = sns.color_palette(["#7428a6", "#fb98dd"])
bcq_dist_sub = bcq_dist[bcq_dist.BCQ_jian_level.isin([2,4])]
d = {2:"L",4:"H"}
bcq_dist_sub["jian_cat"] = bcq_dist_sub.BCQ_jian_level.apply(lambda x: d[x])

In [ ]:
sns.set(font="Times New Roman", style="white")
fontsize = 18
fig, ax = plt.subplots(1, figsize=(8,8))
sns.barplot(x="jian_cat",y="cnt", data=bcq_dist_sub[bcq_dist_sub["Attractive level"]=="More Attractive"],palette=palette, ax=ax)
ax.set_title("More Attractive Hosts",fontsize=fontsize, fontweight="bold")
ax.tick_params(labelsize=fontsize)
ax.set_xlabel("Discount Level", fontsize=fontsize, fontweight="bold",fontname="Times New Roman")
ax.set_ylabel("Number of Coupons", fontsize=fontsize, fontweight="bold",fontname="Times New Roman")
# plt.show()
plt.savefig('./bcq_high_attractive.pdf')
plt.tight_layout()

In [ ]:
fontsize = 18
fig, ax = plt.subplots(1, figsize=(8,8))
sns.barplot(x="jian_cat",y="cnt", data=bcq_dist_sub[bcq_dist_sub["Attractive level"]=="Less Attractive"],palette=palette, ax=ax)
ax.set_title("Less Attractive Hosts",fontsize=fontsize, fontweight="bold")
ax.tick_params(labelsize=fontsize)
ax.set_xlabel("Discount Level", fontsize=fontsize, fontweight="bold",fontname="Times New Roman")
ax.set_ylabel("Number of Coupons", fontsize=fontsize, fontweight="bold",fontname="Times New Roman")
# plt.show()
plt.savefig('bcq_low_attractive.pdf')
plt.tight_layout()

# Figure 12 See Excel
# Figure 13 See Excel

# Figure A1: Evidence of Consumer Exogenous Search Behavior

## A1 (a)

In [ ]:
temp =pd.concat([df[["buyer_id","receive_time_id","buyer_stat_dim_tb_mbr_tq_score","prod_stat_avg_reserve_price","host_liv_stat_alipay_amt","host_liv_stat_rate_sum","host_liv_stat_seller_service_score","host_liv_stat_sp_starts_time","host_liv_stat_fans_value_score","host_liv_stat_read_time_avg"]],action_text],axis=1)
temp1=temp.merge(action_recategorize,left_on='level_1',right_on='text')
temp2=temp1.sort_values(['buyer_id','receive_time_id'])
temp2['jian_lag'] = temp2.groupby('buyer_id')['jian_cat'].shift(1)
temp2['man_lag'] = temp2.groupby('buyer_id')['man_cat'].shift(1)
temp2['rating_lag'] = temp2.groupby('buyer_id')['host_liv_stat_rate_sum'].shift(1)
temp2['revisit'] = np.where(temp2['host_liv_stat_rate_sum']== temp2['rating_lag'], 1, 0)
df_plot = temp2.copy()
df_plot_trim_only2=df_plot[df_plot['man_lag'] ==3]

In [ ]:
font = {'fontname':'Times New Roman'}
fig,ax=plt.subplots(figsize=(5,5))
fontsize=18
sns.barplot(x="jian_lag", y="host_liv_stat_alipay_amt", data=df_plot_trim_only2, ci=99, errwidth=1,capsize=0.1,palette="Purples")   
ax.set_xlabel('Discount Level of Coupon Received at t',fontsize=fontsize, **font)
ax.set_ylabel('Sales of Livestream Channel Visited at t+1',fontsize=fontsize,**font)
ax.tick_params(labelsize=18)
for tick in ax.get_xticklabels():
    tick.set_fontname('Times New Roman')
for tick in ax.get_yticklabels():
    tick.set_fontname('Times New Roman')
plt.xticks([0,1, 2, 3 ,4],labels=['LL', 'L', 'M','H','HH'])
ax.set_ylim([-0.3,0.3])
plt.tight_layout()
plt.savefig('search_sales_man3.pdf')
plt.show()

## A1 (b)

In [ ]:
font = {'fontname':'Times New Roman'}
fig,ax=plt.subplots(figsize=(5,5))
fontsize=18
sns.barplot(x="jian_lag", y="host_liv_stat_seller_service_score", data=df_plot_trim_only2, ci=99, errwidth=1,capsize=0.1,palette="Purples")   
ax.set_xlabel('Discount Level of Coupon Received at t',fontsize=fontsize, **font)
ax.set_ylabel('Rating of Livestream Channel Visited at t+1',fontsize=fontsize,**font)
ax.tick_params(labelsize=18)
for tick in ax.get_xticklabels():
    tick.set_fontname('Times New Roman')
for tick in ax.get_yticklabels():
    tick.set_fontname('Times New Roman')
plt.xticks([0,1, 2, 3 ,4],labels=['LL', 'L', 'M','H','HH'])
ax.set_ylim([-0.3,0.3])
plt.tight_layout()
plt.savefig('search_service_rating_man3.pdf')
plt.show()

## A1 (c)

In [ ]:
font = {'fontname':'Times New Roman'}
fig,ax=plt.subplots(figsize=(5,5))
fontsize=18
sns.barplot(x="jian_lag", y="prod_stat_avg_reserve_price", data=df_plot_trim_only2, ci=99, errwidth=1,capsize=0.1,palette="Purples")   
ax.set_xlabel('Discount Level of Coupon Received at t',fontsize=fontsize, **font)
ax.set_ylabel('Price of Livestream Product Visited at t+1',fontsize=fontsize,**font)
ax.tick_params(labelsize=18)
for tick in ax.get_xticklabels():
    tick.set_fontname('Times New Roman')
for tick in ax.get_yticklabels():
    tick.set_fontname('Times New Roman')
plt.xticks([0,1, 2, 3 ,4],labels=['LL', 'L', 'M','H','HH'])
ax.set_ylim([-0.05,0.05])
plt.tight_layout()
plt.savefig('search_price_man3.pdf')
plt.show()

## A1 (d)

In [ ]:
font = {'fontname':'Times New Roman'}
fig,ax=plt.subplots(figsize=(5,5))
fontsize=18
sns.barplot(x="jian_lag", y="revisit", data=df_plot_trim_only2, ci=99, errwidth=1,capsize=0.1,palette="Purples")   
ax.set_xlabel('Discount Level of Coupon Received at t',fontsize=fontsize, **font)
ax.set_ylabel('Revisit Probability at t+1',fontsize=fontsize,**font)
ax.tick_params(labelsize=18)
for tick in ax.get_xticklabels():
    tick.set_fontname('Times New Roman')
for tick in ax.get_yticklabels():
    tick.set_fontname('Times New Roman')
plt.xticks([0,1, 2, 3 ,4],labels=['LL', 'L', 'M','H','HH'])
#ax.set_ylim([-0.05,0.05])
plt.tight_layout()
plt.savefig('search_revisit_man3.pdf')
plt.show()

# Figure A2: Coupon Redeption Rate by Prior Purchase Experience and Purchase Frequency

## A2 (a)

In [ ]:
Pur_Freq = temp2.groupby('buyer_id').agg(buy_sum=('buy', 'sum')).reset_index()

In [ ]:
Inf_buyer = Pur_Freq[Pur_Freq['buy_sum']<5]
Freq_buyer = Pur_Freq[Pur_Freq['buy_sum']>=5]

In [ ]:
Inf_buyer_obs = temp2[temp2.buyer_id.isin(Inf_buyer.buyer_id)]
Freq_buyer_obs = temp2[temp2.buyer_id.isin(Freq_buyer.buyer_id)]

In [ ]:
# Select IDs two consecutive H coupons
temp_1 = Inf_buyer_obs[(Inf_buyer_obs['jian_cat']==4) & (Inf_buyer_obs['jian_lag']==4)]
temp_1['period'] = "t"
temp_2 = Inf_buyer_obs[(Inf_buyer_obs['jian_lead']==4) & (Inf_buyer_obs['jian_cat']==4)]
temp_2['period'] = "t-1"
frames = [temp_1, temp_2]
temp_combine = pd.concat(frames)
temp_combine = temp_combine.sort_values(by=['buyer_id', 'receive_time_id'])

In [ ]:
lst = [['No_Purchase_t-1', temp_combine.buy[temp_combine["buy_lag"] == 0].mean(),temp_combine.buy[temp_combine["buy_lag"] == 0].sem()], 
       ['Purchase_t-1'   , temp_combine.buy[temp_combine["buy_lag"] == 1].mean(),temp_combine.buy[temp_combine["buy_lag"] == 1].sem()]      ]   
vs_light= pd.DataFrame(lst, columns =['label', 'mean','sem'])

In [ ]:
palette = sns.color_palette(['#7428a6'])
sns.set(font="Times New Roman", style="white")
fontsize = 18
fig, ax = plt.subplots(figsize=(5, 5))
ax.set(ylim=(0, 0.25))
sns.barplot(x=vs_light["label"],y=vs_light["mean"], ax=ax,palette=palette)
plt.errorbar(x=vs_light["label"],y=vs_light["mean"],yerr=vs_light["sem"], fmt='none',c ="black",elinewidth=3, capsize=10)
ax.tick_params(labelsize=fontsize)
ax.set_xlabel('', fontsize=fontsize, fontweight="bold",fontname ='Times New Roman')
ax.set_ylabel('Pruchase_Likelihood_t', fontsize=fontsize, fontweight="bold",fontname ='Times New Roman')
plt.tight_layout()
#plt.show()
plt.savefig('./variety_seeking_HH_light_buy_paper.pdf')

## A2 (b)

In [ ]:
# Select IDs two consecutive H coupons
temp_1 = Freq_buyer_obs[(Freq_buyer_obs['jian_cat']==4) & (Freq_buyer_obs['jian_lag']==4)]
temp_1['period'] = "t"
temp_2 = Freq_buyer_obs[(Freq_buyer_obs['jian_lead']==4) & (Freq_buyer_obs['jian_cat']==4)]
temp_2['period'] = "t-1"
frames = [temp_1, temp_2]
temp_combine = pd.concat(frames)
temp_combine = temp_combine.sort_values(by=['buyer_id', 'receive_time_id'])

In [ ]:
lst = [['No_Purchase_t-1', temp_combine.buy[temp_combine["buy_lag"] == 0].mean(),temp_combine.buy[temp_combine["buy_lag"] == 0].sem()], 
       ['Purchase_t-1'   , temp_combine.buy[temp_combine["buy_lag"] == 1].mean(),temp_combine.buy[temp_combine["buy_lag"] == 1].sem()]      ]   
vs_heavy = pd.DataFrame(lst, columns =['label', 'mean','sem'])

In [ ]:
palette = sns.color_palette(['#7428a6'])
sns.set(font="Times New Roman", style="white")
fontsize = 18
fig, ax = plt.subplots(figsize=(5, 5))
ax.set(ylim=(0, 0.25))
sns.barplot(x=vs_heavy["label"],y=vs_heavy["mean"], ax=ax,palette=palette)
plt.errorbar(x=vs_heavy["label"],y=vs_heavy["mean"],yerr=vs_heavy["sem"], fmt='none',c ="black",elinewidth=3, capsize=10)
ax.tick_params(labelsize=fontsize)
ax.set_xlabel('', fontsize=fontsize, fontweight="bold",fontname ='Times New Roman')
ax.set_ylabel('Pruchase_Likelihood_t', fontsize=fontsize, fontweight="bold",fontname ='Times New Roman')
plt.tight_layout()
#plt.show()
plt.savefig('./variety_seeking_HH_heavy_buy_paper.pdf')

# Figure A4: Illustration of Model Bias

In [ ]:
def scatterWline(x_scatter,y_scatter,color):
    x_scatter,y_scatter  = np.array(x_scatter),np.array(y_scatter)
    plt.scatter(x_scatter,y_scatter,color=color)
    b, m = polyfit(x_scatter, y_scatter, 1)
    xlst = np.arange(0,10,0.1)
    plt.plot(xlst, b + m * xlst, '-',color=color)
    return b,m

def left(x):
    return 0.4*(x-6)**2+8

def right(x):
    return 0.4*(x-4)**2+7

x_lst=list(np.arange(0,10,0.1))
y1_lst=[ left(x) for x in x_lst]
y2_lst=[ right(x) for x in x_lst]
x1_scatter =list(10-np.exp(np.arange(-12,2.12,0.11)))
y1_scatter=[left(x) for x in x1_scatter]
x2_scatter = [10-x for x in x1_scatter]
y2_scatter=[right(x) for x in x2_scatter]

In [ ]:
fig = plt.figure()
axes=fig.add_subplot(111)
xmin, xmax, ymin, ymax = 0,10,0,20
axes.set(xlim=(xmin, xmax), ylim=(ymin, ymax))
plt.plot(x_lst,y2_lst,color ='g',label="High Discount Coupon")
plt.plot(x_lst,y1_lst,color ='r',label="Low Discount Coupon")
b1,m1 = scatterWline(x1_scatter,y1_scatter,'r')
b2,m2 = scatterWline(x2_scatter,y2_scatter,'g')
plt.xlabel('Loyalty')
plt.ylabel('Revenue')
plt.legend(loc="upper left")
plt.savefig('./without_bias.pdf')
plt.show()

In [ ]:
fig = plt.figure()
axes=fig.add_subplot(111)
xmin, xmax, ymin, ymax = 0,10,0,20
axes.set(xlim=(xmin, xmax), ylim=(ymin, ymax))
plt.plot(x_lst,y2_lst,color ='g',label="High Discount Coupon")
plt.plot(x_lst,y1_lst,color ='r',label="Low Discount Coupon")

x1_scatter = [x for x in list(1.5+np.exp(np.arange(-0.5,0.9,0.15)))]
y1_scatter=[left(x) for x in x1_scatter]
x2_scatter =[10-x for x in x1_scatter]
y2_scatter=[right(x) for x in x2_scatter]


plt.scatter(x1_scatter,y1_scatter,color='r')
plt.scatter(x2_scatter,y2_scatter,color='g')
xlst = np.arange(0,10,0.1)
plt.plot(xlst, b1 + m1 * xlst, '-',color='r')
plt.plot(xlst, b2 + m2 * xlst, '-',color='g')

for i in range(len(x1_scatter)):
    plt.vlines(x=x1_scatter[i], ymin=y1_scatter[i], ymax=b1 + m1 * x1_scatter[i], color='r')
for i in range(len(x2_scatter)):
    plt.vlines(x=x2_scatter[i], ymin=y2_scatter[i], ymax=b2 + m2 * x2_scatter[i], color='g')
plt.xlabel('Loyalty')
plt.ylabel('Revenue')
plt.legend(loc="upper left")
plt.savefig('./with_bias.pdf')
plt.show()

# Figure A5: R
# Figure A6: R

# Figure A7: Current and Future Redemption Rates by Coupon Discount Level and User Activity Level

In [ ]:
coupon_freq = temp2.groupby('buyer_id')["receive_time_id"].count()
coupon_freq = coupon_freq.to_frame()

In [ ]:
## Reference Price by inactive vs active users

## A7(a)

In [ ]:
#inactive
coupon_freq_remove = coupon_freq[coupon_freq.receive_time_id >=25]
common = temp2.merge(coupon_freq_remove,on=['buyer_id','buyer_id'])
temp4 = temp2[(~temp2.buyer_id.isin(common.buyer_id))&(~temp2.buyer_id.isin(common.buyer_id))]
#print(temp4.shape)
temp_1 = temp4[temp4['jian_cat']==3]
temp_1['period'] = "RedemptionRate_M_t+1"
temp_2 = temp4[temp4['jian_lead']==3]
temp_2['period'] = "RedemptionRate_t"
frames = [temp_1, temp_2]
temp_combine = pd.concat(frames)
temp_combine = temp_combine.sort_values(by=['buyer_id', 'receive_time_id'])

In [ ]:
temp_combine['coupon_t-1'] = np.where(temp_combine['period']== 't-1', temp_combine['jian_cat'], temp_combine['jian_lag'])
tempplot = temp_combine.groupby(['coupon_t-1','period']).agg({'buy': ['mean', 'sem']}).xs('buy', axis=1, drop_level=True).reset_index(['coupon_t-1','period']).sort_values(['coupon_t-1', 'period'], ascending=[True, False])
d = {1:"LL",2:"L",3:"M",4:"H",5:"HH"}
tempplot["Discount level"] = tempplot["coupon_t-1"].apply(lambda x:d[x])
tempplot = tempplot[(tempplot['coupon_t-1']==2) | (tempplot['coupon_t-1']==4)]

In [ ]:
sns.set(font="Times New Roman", style="white")
palette = sns.color_palette(['#fb98dd','#7428a6'])
fig, ax = plt.subplots(figsize=(5, 5),dpi=100)
plt.ylim(0, 0.15)
sns.barplot(x='Discount level', y='mean', 
                hue="period", data=tempplot, ax=ax, palette=palette)

ax.tick_params(labelsize=fontsize)
ax.set_xlabel('Coupon Discount Level t', fontsize=fontsize,fontweight="bold",fontname ='Times New Roman')
ax.set_ylabel('Redemption Rate', fontsize=fontsize,fontweight="bold",fontname ='Times New Roman')

leg = ax.legend(fontsize=fontsize)
new_title = ""
leg.set_title(new_title,prop={'size':fontsize})
plt.tight_layout()
#plt.show()
plt.savefig('./ref_price_LH_light_paper.pdf')

## A7(b)

In [ ]:
#active
coupon_freq_remove = coupon_freq[coupon_freq.receive_time_id <25]
common = temp2.merge(coupon_freq_remove,on=['buyer_id','buyer_id'])
temp4 = temp2[(~temp2.buyer_id.isin(common.buyer_id))&(~temp2.buyer_id.isin(common.buyer_id))]
#print(temp4.shape)
temp_1 = temp4[temp4['jian_cat']==3]
temp_1['period'] = "RedemptionRate_M_t+1"
temp_2 = temp4[temp4['jian_lead']==3]
temp_2['period'] = "RedemptionRate_t"
frames = [temp_1, temp_2]
temp_combine = pd.concat(frames)
temp_combine = temp_combine.sort_values(by=['buyer_id', 'receive_time_id'])

In [ ]:
temp_combine['coupon_t-1'] = np.where(temp_combine['period']== 't-1', temp_combine['jian_cat'], temp_combine['jian_lag'])
tempplot = temp_combine.groupby(['coupon_t-1','period']).agg({'buy': ['mean', 'sem']}).xs('buy', axis=1, drop_level=True).reset_index(['coupon_t-1','period']).sort_values(['coupon_t-1', 'period'], ascending=[True, False])
d = {1:"LL",2:"L",3:"M",4:"H",5:"HH"}
tempplot["Discount level"] = tempplot["coupon_t-1"].apply(lambda x:d[x])
tempplot = tempplot[(tempplot['coupon_t-1']==2) | (tempplot['coupon_t-1']==4)]

In [ ]:
sns.set(font="Times New Roman", style="white")
palette = sns.color_palette(['#fb98dd','#7428a6'])
fig, ax = plt.subplots(figsize=(5, 5),dpi=100)
plt.ylim(0, 0.16)
fontsize = 18
sns.barplot(x='Discount level', y='mean', 
                hue="period", data=tempplot, ax=ax, palette=palette)

ax.tick_params(labelsize=fontsize)
ax.set_xlabel('Coupon Discount Level t', fontsize=fontsize,fontweight="bold",fontname ='Times New Roman')
ax.set_ylabel('Redemption Rate', fontsize=fontsize,fontweight="bold",fontname ='Times New Roman')

leg = ax.legend(fontsize=fontsize)
new_title = ""
leg.set_title(new_title,prop={'size':fontsize})
plt.tight_layout()
#plt.show()
plt.savefig('./ref_price_LH_heavy_paper.pdf')

# Figure A8: Current and Future Redemption Rates by Prior Experience and User Activity Level

## A8(a) Inactive Users

In [ ]:
coupon_freq_remove = coupon_freq[coupon_freq.receive_time_id >=25]
common = temp2.merge(coupon_freq_remove,on=['buyer_id','buyer_id'])
temp3 = temp2[(~temp2.buyer_id.isin(common.buyer_id))&(~temp2.buyer_id.isin(common.buyer_id))]
#print(temp4.shape)
# Select IDs two consecutive H coupons
temp_1 = temp3[(temp3['jian_cat']==4) & (temp3['jian_lag']==4)]
temp_1['period'] = "t"
temp_2 = temp3[(temp3['jian_lead']==4) & (temp3['jian_cat']==4)]
temp_2['period'] = "t-1"
frames = [temp_1, temp_2]
temp_combine = pd.concat(frames)
temp_combine = temp_combine.sort_values(by=['buyer_id', 'receive_time_id'])

In [ ]:
lst = [['No_Purchase_t-1', temp_combine.buy[temp_combine["buy_lag"] == 0].mean(),temp_combine.buy[temp_combine["buy_lag"] == 0].sem()], 
       ['Purchase_t-1'   , temp_combine.buy[temp_combine["buy_lag"] == 1].mean(),temp_combine.buy[temp_combine["buy_lag"] == 1].sem()]      ]   
vs_light = pd.DataFrame(lst, columns =['label', 'mean','sem'])

In [ ]:
palette = sns.color_palette(['#7428a6'])
sns.set(font="Times New Roman", style="white")
fontsize = 18
fig, ax = plt.subplots(figsize=(5, 5))
ax.set(ylim=(0, 0.25))
sns.barplot(x=vs_light["label"],y=vs_light["mean"], ax=ax,palette=palette)
plt.errorbar(x=vs_light["label"],y=vs_light["mean"],yerr=vs_light["sem"], fmt='none',c ="black",elinewidth=3, capsize=10)
ax.tick_params(labelsize=fontsize)
ax.set_xlabel('', fontsize=fontsize, fontweight="bold",fontname ='Times New Roman')
ax.set_ylabel('Pruchase_Likelihood_t', fontsize=fontsize, fontweight="bold",fontname ='Times New Roman')
plt.tight_layout()
#plt.show()
plt.savefig('./variety_seeking_HH_light_visit_paper.pdf')

## A8(b) Active

In [ ]:
lst = [['No_Purchase_t-1', temp_combine.buy[temp_combine["buy_lag"] == 0].mean(),temp_combine.buy[temp_combine["buy_lag"] == 0].sem()], 
       ['Purchase_t-1'   , temp_combine.buy[temp_combine["buy_lag"] == 1].mean(),temp_combine.buy[temp_combine["buy_lag"] == 1].sem()]      ]   
vs_heavy = pd.DataFrame(lst, columns =['label', 'mean','sem'])

In [ ]:
palette = sns.color_palette(['#7428a6'])
sns.set(font="Times New Roman", style="white")
fontsize = 18
fig, ax = plt.subplots(figsize=(5, 5))
ax.set(ylim=(0, 0.25))
sns.barplot(x=vs_heavy["label"],y=vs_heavy["mean"], ax=ax,palette=palette)
plt.errorbar(x=vs_heavy["label"],y=vs_heavy["mean"],yerr=vs_heavy["sem"], fmt='none',c ="black",elinewidth=3, capsize=10)
ax.tick_params(labelsize=fontsize)
ax.set_xlabel('', fontsize=fontsize, fontweight="bold",fontname ='Times New Roman')
ax.set_ylabel('Pruchase_Likelihood_t', fontsize=fontsize, fontweight="bold",fontname ='Times New Roman')
plt.tight_layout()
#plt.show()
plt.savefig('./variety_seeking_HH_heavy_visit_paper.pdf')

# Figure A9: Targeting Rule Under BDRL (Undiscounted Case): Host Attractiveness

In [ ]:
temp5 = pd.read_csv('./BDRL_less_attractive.csv', delimiter= ',')

In [ ]:
sns.set(font="Times New Roman", style="white")
fontsize = 18
palette = sns.color_palette(["#7428a6", "#fb98dd"])
fig, ax = plt.subplots(1, figsize=(5,5))
sns.barplot(x="jian_cat",y="cnt", data=temp5,palette=palette, ax=ax)
ax.set_title("Less Attractive Hosts",fontsize=fontsize, fontweight="bold")
ax.ticklabel_format(style='plain', axis='y')
ax.tick_params(labelsize=12)
ax.set_xlabel("Discount Level", fontsize=fontsize, fontweight="bold",fontname="Times New Roman")
ax.set_ylabel("Number of Coupons", fontsize=fontsize, fontweight="bold",fontname="Times New Roman")
# plt.show()
plt.tight_layout()
plt.savefig('./bcq_low_attractive.pdf')

In [ ]:
temp6 = pd.read_csv('./BDRL_more_attractive.csv', delimiter= ',')

In [ ]:
fontsize = 18
fig, ax = plt.subplots(1, figsize=(5,5))
sns.barplot(x="jian_cat",y="cnt", data=temp6,palette=palette, ax=ax)
ax.set_title("More Attractive Hosts",fontsize=fontsize, fontweight="bold")
ax.tick_params(labelsize=fontsize)
ax.ticklabel_format(style='plain', axis='y')
ax.set_xlabel("Discount Level", fontsize=fontsize, fontweight="bold",fontname="Times New Roman")
ax.set_ylabel("Number of Coupons", fontsize=fontsize, fontweight="bold",fontname="Times New Roman")
# plt.show()
plt.tight_layout()
plt.savefig('./bcq_high_attractive.pdf')

# Figure A10: See Excel
# Figure A11: See Excel

# Table 2: Coupon Discretization by Threshold Ratio and Discount Ratio

In [ ]:
temp2.man.quantile([.2,.4 .6,.8, 1.0])
temp2.jian.quantile([.2,.4 .6,.8, 1.0])

# Table A1: No Forward Looking

1. find a subset of consumers whose discount sequence is upward, 
2. compare their interpurchase time with another subset whose discount sequence is flat, 
3. show that the upward people's interpurchase time is smaller than that of flat people;

In [ ]:
temp2['diff']=temp2['jian_cat'].diff()
mask=temp2.buyer_id != temp2.buyer_id.shift(1)
temp2['diff'][mask]=np.nan

# Upward Sequence

In [ ]:
# Upward Sequence 
temp3=temp2.groupby(['buyer_id'])['diff'].min().to_frame()
temp3.columns=['diff_min']
temp3

In [ ]:
# Downward Sequence
temp4=temp2.groupby(['buyer_id'])['diff'].max().to_frame()
temp4.columns=['diff_max']

In [ ]:
min_max=pd.concat([temp4,temp3],axis=1)
min_max

In [ ]:
# IDs with upward tedency
up_id=min_max.index[(min_max['diff_min']>=0)&(min_max['diff_max']>0)]
up_id

In [ ]:
# Select upward IDs from data
up=temp2.loc[up_id].reset_index() 

In [ ]:
# Calculate interpurchase time
up_purchase_time=up.groupby('buyer_id')['buy'].sum().reset_index()
up_purchase_time.columns=['buyer_id','Purchase Time']
up_purchase_time

In [ ]:
# Calculate Number of buyers, Mean , Std
count_up=up_purchase_time['buyer_id'].count()
mean_up=up_purchase_time['Purchase Time'].mean()
std_up=up_purchase_time['Purchase Time'].std()
print("There are {} Upward buyers. Mean is {}. Standard deviation is {}.".format(count_up,mean_up,std_up))

# Downward Sequence

In [ ]:
# Select IDs with downward tedency
down_id=min_max.index[(min_max['diff_max']<=0)&(min_max['diff_min']<0)]
down=temp2.loc[down_id].reset_index()

In [ ]:
down_purchase_time=down.groupby('buyer_id')['buy'].sum().reset_index()
down_purchase_time.columns=['buyer_id','Purchase Time']
down_purchase_time

In [ ]:
# Calculate Number of buyers, Mean , Std
count_down=down_purchase_time['buyer_id'].count()
mean_down=down_purchase_time['Purchase Time'].mean()
std_down=down_purchase_time['Purchase Time'].std()
print("There are {} Downward buyers. Mean is {}. Standard deviation is {}.".format(count_down,mean_down,std_down))

# Flat

In [ ]:
# Flat (no change) Sequence
flat_id=min_max.index[(min_max['diff_max']==0)&(min_max['diff_min']==0)]
flat=temp2.loc[flat_id].reset_index()

In [ ]:
flat_purchase_time=flat.groupby('buyer_id')['buy'].sum().reset_index()
flat_purchase_time.columns=['buyer_id','Purchase Time']
flat_purchase_time

In [ ]:
# Calculate Number of buyers, Mean , Std
count_flat=flat_purchase_time['buyer_id'].count()
mean_flat=flat_purchase_time['Purchase Time'].mean()
std_flat=flat_purchase_time['Purchase Time'].std()
print("There are {} Flat buyers. Mean is {}. Standard deviation is {}.".format(count_flat,mean_flat,std_flat))

# Control for Discount Level

In [ ]:
up_first=up.groupby('buyer_id')['jian_cat'].first().reset_index()
up_first.columns=['buyer_id','Jian_First']
down_first=down.groupby('buyer_id')['jian_cat'].first().reset_index()
down_first.columns=['buyer_id','Jian_First']
flat_first=flat.groupby('buyer_id')['jian_cat'].first().reset_index()
flat_first.columns=['buyer_id','Jian_First']

In [ ]:
up_first=up_first.set_index(['buyer_id'])
down_first=down_first.set_index(['buyer_id'])
flat_first=flat_first.set_index(['buyer_id'])

In [ ]:
up=up.set_index(['buyer_id'])
down=down.set_index(['buyer_id'])
flat=flat.set_index(['buyer_id'])

## Starting Discount Level = 2

In [ ]:
up_2_id=up_first.index[up_first['Jian_First']==2]
up_2=up.loc[up_2_id].reset_index()
up_2_purchase_time=up_2.loc[up_2['jian_cat']==2].groupby('buyer_id')['buy'].mean().reset_index()
up_2_purchase_time.columns=['buyer_id','Purchase Time']
down_2_id=down_first.index[down_first['Jian_First']==2]
down_2=down.loc[down_2_id].reset_index()
down_2_purchase_time=down_2.loc[down_2['jian_cat']==2].groupby('buyer_id')['buy'].mean().reset_index()
down_2_purchase_time.columns=['buyer_id','Purchase Time']
flat_2_id=flat_first.index[flat_first['Jian_First']==2]
flat_2=flat.loc[flat_2_id].reset_index()
flat_2_purchase_time=flat_2.loc[flat_2['jian_cat']==2].groupby('buyer_id')['buy'].mean().reset_index()
flat_2_purchase_time.columns=['buyer_id','Purchase Time']

In [ ]:
count_up_2=up_2_purchase_time['buyer_id'].count()
mean_up_2=up_2_purchase_time['Purchase Time'].mean()
std_up_2=up_2_purchase_time['Purchase Time'].std()
print("There are {} Up buyers. Mean is {}. Standard deviation is {}.".format(count_up_2,mean_up_2,std_up_2))
count_down_2=down_2_purchase_time['buyer_id'].count()
mean_down_2=down_2_purchase_time['Purchase Time'].mean()
std_down_2=down_2_purchase_time['Purchase Time'].std()
print("There are {} Down buyers. Mean is {}. Standard deviation is {}.".format(count_down_2,mean_down_2,std_down_2))
count_flat_2=flat_2_purchase_time['buyer_id'].count()
mean_flat_2=flat_2_purchase_time['Purchase Time'].mean()
std_flat_2=flat_2_purchase_time['Purchase Time'].std()
print("There are {} Flat buyers. Mean is {}. Standard deviation is {}.".format(count_flat_2,mean_flat_2,std_flat_2))

In [ ]:
# T-TEST - Up and Flat
ttest_ind_from_stats(mean1=mean_up_2, std1=std_up_2, nobs1=count_up_2, mean2=mean_flat_2, std2=std_flat_2, nobs2=count_flat_2)

In [ ]:
# T-TEST - Down and Flat
ttest_ind_from_stats(mean1=mean_down_2, std1=std_down_2, nobs1=count_down_2, mean2=mean_flat_2, std2=std_flat_2, nobs2=count_flat_2)

## Starting Discount Level = 4

In [ ]:
up_4_id=up_first.index[up_first['Jian_First']==4]
up_4=up.loc[up_4_id].reset_index()
up_4_purchase_time=up_3.loc[up_3['jian_cat']==4].groupby('buyer_id')['buy'].mean().reset_index()
up_4_purchase_time.columns=['buyer_id','Purchase Time']
down_4_id=down_first.index[down_first['Jian_First']==4]
down_4=down.loc[down_4_id].reset_index()
down_4_purchase_time=down_4.loc[down_4['jian_cat']==4].groupby('buyer_id')['buy'].mean().reset_index()
down_4_purchase_time.columns=['buyer_id','Purchase Time']
flat_4_id=flat_first.index[flat_first['Jian_First']==4]
flat_4=flat.loc[flat_4_id].reset_index()
flat_4_purchase_time=flat_4.loc[flat_4['jian_cat']==4].groupby('buyer_id')['buy'].mean().reset_index()
flat_4_purchase_time.columns=['buyer_id','Purchase Time']

In [ ]:
count_up_4=up_4_purchase_time['buyer_id'].count()
mean_up_4=up_4_purchase_time['Purchase Time'].mean()
std_up_4=up_4_purchase_time['Purchase Time'].std()
print("There are {} Up buyers. Mean is {}. Standard deviation is {}.".format(count_up_4,mean_up_4,std_up_4))
count_down_4=down_4_purchase_time['buyer_id'].count()
mean_down_4=down_4_purchase_time['Purchase Time'].mean()
std_down_4=down_4_purchase_time['Purchase Time'].std()
print("There are {} Down buyers. Mean is {}. Standard deviation is {}.".format(count_down_4,mean_down_4,std_down_4))
count_flat_4=flat_4_purchase_time['buyer_id'].count()
mean_flat_4=flat_4_purchase_time['Purchase Time'].mean()
std_flat_4=flat_4_purchase_time['Purchase Time'].std()
print("There are {} Flat buyers. Mean is {}. Standard deviation is {}.".format(count_flat_4,mean_flat_4,std_flat_4))

In [ ]:
# T-TEST - Up and Flat
ttest_ind_from_stats(mean1=mean_up_4, std1=std_up_4, nobs1=count_up_4, mean2=mean_flat_4, std2=std_flat_4, nobs2=count_flat_4)

In [ ]:
# T-TEST - Down and Flat
ttest_ind_from_stats(mean1=mean_down_4, std1=std_down_4, nobs1=count_down_4, mean2=mean_flat_4, std2=std_flat_4, nobs2=count_flat_4)

# Table A5: Summary Statistics of State Variables

In [ ]:
#TQZ
df['buyer_stat_dim_tb_mbr_tq_score'].describe()
#Frequency coupon
df['buyer_dynamic_num_receive'].describe()
#Attractiveness
df['host_liv_stat_fans_value_score'].describe()
#Price
df['prod_stat_avg_reserve_price'].describe()
# Times add to cart
df['host_liv_stat_pv_cart'].describe()
# Frequency product
df['buyer_dynamic_num_transactions'].describe()